In [3]:
import httpx
import json

# --- 1. CONFIGURATION ---
# Change these values to match your specific setup
ERPNEXT_URL = "https://erpnext-ai-integration.m.frappe.cloud"  # e.g., https://mycompany.com
EMAIL = "gharadpatil75@gmail.com"
PASSWORD = "@Mahendra@123"

print(f"🔄 Attempting to connect to ERPNext instance at: {ERPNEXT_URL}...\n")

# Use an asynchronous or synchronous client. For Jupyter notebooks, 
# a synchronous client (httpx.Client) is easiest to run inline.
with httpx.Client(timeout=15.0) as client:
    
    # --- STEP 2: AUTHENTICATE & GET COOKIES ---
    login_url = f"{ERPNEXT_URL}/api/method/login"
    login_payload = {
        "usr": EMAIL,
        "pwd": PASSWORD
    }
    
    print("Step 1: Sending email and password to ERPNext...")
    login_response = client.post(login_url, data=login_payload)
    
    if login_response.status_code != 200 or login_response.json().get("message") != "Logged In":
        print("❌ Login Failed!")
        print(f"Status Code: {login_response.status_code}")
        print(f"Response Body: {login_response.text}")
        raise ValueError("Stop execution: Check your URL, Email, or Password.")
        
    print("✅ Successfully logged in via session cookies!")
    print(f"Returned Message: {login_response.json()}")
    print("-" * 50)

    # --- STEP 3: DYNAMICALLY GENERATE/FETCH API KEYS ---
    # The client automatically handles the session cookies set in Step 2.
    generate_keys_url = f"{ERPNEXT_URL}/api/method/frappe.core.doctype.user.user.generate_keys"
    
    print("Step 2: Requesting User API Keys from ERPNext...")
    keys_response = client.post(generate_keys_url)
    
    if keys_response.status_code == 200:
        keys_data = keys_response.json().get("message", {})
        api_key = keys_data.get("api_key")
        api_secret = keys_data.get("api_secret")
        
        print("🎉 SUCCESS! API Credentials Received:")
        print(f"   🔹 API Key:    {api_key}")
        print(f"   🔹 API Secret: {api_secret}")
        print("\n💡 You can now use these stateless tokens in your FastAPI bridge!")
    else:
        print("⚠️ Direct API generation endpoint skipped or not allowed. Trying fallback...")
        
        # FALLBACK: Try pulling standard logged-in user profile info
        fallback_url = f"{ERPNEXT_URL}/api/method/frappe.auth.get_logged_user"
        fallback_response = client.get(fallback_url)
        
        print(f"Status Code: {fallback_response.status_code}")
        print(f"Profile Check Response: {fallback_response.text}")


🔄 Attempting to connect to ERPNext instance at: https://erpnext-ai-integration.m.frappe.cloud...

Step 1: Sending email and password to ERPNext...
✅ Successfully logged in via session cookies!
Returned Message: {'message': 'Logged In', 'home_page': 'desk', 'full_name': 'Mahendra Gharad'}
--------------------------------------------------
Step 2: Requesting User API Keys from ERPNext...
⚠️ Direct API generation endpoint skipped or not allowed. Trying fallback...
Status Code: 200
Profile Check Response: {"message":"gharadpatil75@gmail.com"}


In [4]:
import httpx

# --- CONFIGURATION ---
ERPNEXT_URL = "https://erpnext-ai-integration.m.frappe.cloud"
API_KEY = "2dd96d29e8eeb24"
API_SECRET = "82281010108cb7e"

# Format the stateless authorization header required by Frappe/ERPNext
headers = {
    "Authorization": f"token {API_KEY}:{API_SECRET}"
}

print("🔄 Testing stateless token communication with ERPNext...")

with httpx.Client(timeout=10.0) as client:
    # Ping the logged-in user profile method using only the tokens
    response = client.get(f"{ERPNEXT_URL}/api/method/frappe.auth.get_logged_user", headers=headers)
    
    if response.status_code == 200:
        print("🎉 SUCCESS! Token is completely valid.")
        print(f"Logged in as: {response.json().get('message')}")
    else:
        print("❌ Token authentication failed.")
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.text}")


🔄 Testing stateless token communication with ERPNext...
🎉 SUCCESS! Token is completely valid.
Logged in as: gharadpatil75@gmail.com


In [7]:
import httpx
import json
from datetime import datetime

# --- CONFIGURATION ---
ERPNEXT_URL = "https://erpnext-ai-integration.m.frappe.cloud"
API_KEY = "2dd96d29e8eeb24"
API_SECRET = "82281010108cb7e"

headers = {
    "Authorization": f"token {API_KEY}:{API_SECRET}",
    "Content-Type": "application/json"
}

# --- PR DOCUMENT PAYLOAD ---
# Note: Customize these field names based on your ERPNext configuration requirements.
pr_payload = {
        "doc": {
            "doctype": "Material Request",
            "material_request_type": "Purchase",
            "transaction_date": "2026-12-10",
            "schedule_date": "2026-12-25",
            "company": "Maahi's AI Learning Corporation",
            "department": "Management",
            "title": "Quarterly IT & Hardware Procurement",
            "remark": "Urgent requisition for upcoming engineering team onboarding.",
            "items": [
            {
                "item_code": "ITEM-0010",
                "item_name": "Testing",
                "description": "High-performance workstation component with extended 3-year hardware warranty.",
                "qty": 15,
                "uom": "Nos",
                "schedule_date": "2026-12-25",
                "warehouse": "All Warehouses - MALC",
                "rate": 2500.00
      }
    ]
  }
}

print("🔄 Attempting to create a Purchase Requisition in ERPNext...")

with httpx.Client(timeout=15.0) as client:
    # ERPNext creation API endpoint format: /api/resource/{DocType}
    url = f"{ERPNEXT_URL}/api/method/frappe.client.insert"
    
    response = client.post(url, headers=headers, json=pr_payload)
    
    if response.status_code == 200:
        created_doc = response.json().get("data", {})
        print("\n🎉 SUCCESS! Purchase Requisition Created.")
        print(f"📄 Document ID: {created_doc.get('name')}")
        print(f"🏢 Company:     {created_doc.get('company')}")
        print(f"📦 Total Items: {len(created_doc.get('items', []))}")
    else:
        print("\n❌ Failed to create Purchase Requisition.")
        print(f"Status Code: {response.status_code}")
        print("Response Text:")
        # Pretty print the server error to diagnose missing required fields
        try:
            print(json.dumps(response.json(), indent=4))
        except:
            print(response.text)


🔄 Attempting to create a Purchase Requisition in ERPNext...

🎉 SUCCESS! Purchase Requisition Created.
📄 Document ID: None
🏢 Company:     None
📦 Total Items: 0


In [10]:
import httpx
from typing import Dict, Any, Optional

class ERPNextClientService:
    def __init__(self, base_url: str):
        """Initializes the service with the target ERPNext domain URL."""
        self.base_url = base_url.rstrip('/')
        # The Client session automatically handles and holds cookies in memory
        self.client = httpx.Client(timeout=15.0)
        self.is_authenticated = False

    def login(self, email: str, password: str) -> bool:
        """Authenticates directly with ERPNext using email and password."""
        url = f"{self.base_url}/api/method/login"
        payload = {
            "usr": email,
            "pwd": password
        }
        
        try:
            response = self.client.post(url, data=payload)
            
            if response.status_code == 200 and response.json().get("message") == "Logged In":
                self.is_authenticated = True
                print(f"✅ Authenticated successfully as {email}")
                return True
            else:
                self.is_authenticated = False
                print(f"❌ Login failed. Status: {response.status_code}, Response: {response.text}")
                return False
        except httpx.RequestError as e:
            print(f"❌ Connection error during login: {str(e)}")
            self.is_authenticated = False
            return False

    def execute(self, method: str, endpoint: str, payload: Optional[Dict[str, Any]] = None, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """
        Executes a dynamic API request to any ERPNext endpoint.
        
        :param method: HTTP Verb (GET, POST, PUT, DELETE)
        :param endpoint: Target path (e.g., 'resource/Purchase Requisition' or 'method/version')
        :param payload: JSON body data for POST/PUT requests
        :param params: URL Query parameters for GET requests
        """
        if not self.is_authenticated:
            raise PermissionError("Client is not authenticated. Call login() successfully first.")

        clean_endpoint = endpoint.lstrip('/')
        full_url = f"{self.base_url}/api/{clean_endpoint}"
        method_upper = method.upper()

        try:
            if method_upper == "GET":
                response = self.client.get(full_url, params=params)
            elif method_upper == "POST":
                response = self.client.post(full_url, json=payload)
            elif method_upper == "PUT":
                response = self.client.put(full_url, json=payload)
            elif method_upper == "DELETE":
                response = self.client.delete(full_url)
            else:
                return {"error": f"Unsupported HTTP method: {method_upper}"}

            # Return parsed JSON response if valid, otherwise return the raw text
            try:
                return response.json()
            except ValueError:
                return {"raw_text": response.text, "status_code": response.status_code}

        except httpx.RequestError as e:
            return {"error": f"Request failed: {str(e)}"}

    def close(self):
        """Closes the underlying HTTP client session connection pool."""
        self.client.close()


# =====================================================================
# HOW TO USE AND TEST THIS SERVICE
# =====================================================================
if __name__ == "__main__":
    
    # 1. Initialize the client service
    erp = ERPNextClientService(base_url="https://erpnext-ai-integration.m.frappe.cloud")

    # 2. Authenticate directly
    login_success = erp.login(email="gharadpatil75@gmail.com", password="@Mahendra@123")

    if login_success:
        # Example Test 1: Run a GET request to see logged-in profile details
        profile_data = erp.execute(method="GET", endpoint="method/frappe.auth.get_logged_user")
        print("\n--- Profile Check Result ---")
        print(profile_data)

        # Example Test 2: Run a POST request to create a Purchase Requisition
        pr_payload = {
                "doc": {
                    "doctype": "Material Request",
                    "material_request_type": "Purchase",
                    "transaction_date": "2026-12-10",
                    "schedule_date": "2026-12-25",
                    "company": "Maahi's AI Learning Corporation",
                    "department": "Management",
                    "title": "Quarterly IT & Hardware Procurement",
                    "remark": "Urgent requisition for upcoming engineering team onboarding.",
                    "items": [
                    {
                        "item_code": "ITEM-0010",
                        "item_name": "Testing",
                        "description": "High-performance workstation component with extended 3-year hardware warranty.",
                        "qty": 15,
                        "uom": "Nos",
                        "schedule_date": "2026-12-25",
                        "warehouse": "All Warehouses - MALC",
                        "rate": 2500.00
            }
            ]
        }
        }
        
        # Uncomment below to test creating actual records
        pr_result = erp.execute(method="POST", endpoint="method/frappe.client.insert", payload=pr_payload)
        print("\n--- PR Creation Result ---")
        print(pr_result)

    # Clean up connection when finished
    erp.close()


✅ Authenticated successfully as gharadpatil75@gmail.com

--- Profile Check Result ---
{'message': 'gharadpatil75@gmail.com'}

--- PR Creation Result ---
{'message': {'name': 'MAT-MR-2026-00007', 'owner': 'gharadpatil75@gmail.com', 'creation': '2026-10-17 14:56:21.378193', 'modified': '2026-10-17 14:56:21.378193', 'modified_by': 'gharadpatil75@gmail.com', 'docstatus': 0, 'idx': 0, 'naming_series': 'MAT-MR-.YYYY.-', 'title': 'Quarterly IT & Hardware Procurement', 'material_request_type': 'Purchase', 'customer': None, 'company': "Maahi's AI Learning Corporation", 'auto_created_via_reorder': 0, 'transaction_date': '2026-12-10', 'schedule_date': '2026-12-25', 'buying_price_list': 'Standard Buying', 'amended_from': None, 'scan_barcode': None, 'last_scanned_warehouse': None, 'set_from_warehouse': None, 'set_warehouse': None, 'tc_name': None, 'terms': None, 'status': 'Draft', 'per_ordered': 0.0, 'transfer_status': '', 'per_received': 0.0, 'letter_head': 'Company Letterhead - Grey', 'select_pri